In [ ]:
SUPER_TOKEN = "PASTE_SHOONYA_SUPER_TOKEN_HERE"
SCAN_DATE = "2026-11-27"

import json
import os
import sys
import time
import urllib.error
import urllib.request
from datetime import datetime, timedelta, time as dt_time
from pathlib import Path

import numpy as np
import pandas as pd
import pytz
from IPython.display import display

BOT_DIR = Path(r"E:\ShoonyaAPI_Code\Updated_Bot")
os.chdir(BOT_DIR)
sys.path.insert(0, str(BOT_DIR))

import config
import fyers_data
import indicators
import shoonya_scanner_bot as bot

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 260)

if SUPER_TOKEN.startswith("PASTE_"):
    raise ValueError("Replace SUPER_TOKEN with the current Shoonya super token.")

SCAN_TIMES = list(config.SCAN_TIMES)
scan_date = datetime.strptime(SCAN_DATE, "%Y-%m-%d").date()
IST = pytz.timezone("Asia/Kolkata")
start_day = scan_date - timedelta(days=7)
start_ep = int(IST.localize(datetime.combine(start_day, dt_time(9, 15))).timestamp())
end_ep = int(IST.localize(datetime.combine(scan_date, dt_time(15, 30))).timestamp())

def bar_time(scan_hm):
    return datetime.strptime("%s %s" % (SCAN_DATE, scan_hm), "%Y-%m-%d %H:%M") - timedelta(minutes=15)

print("PYTHON", sys.executable)
print("SCAN_DATE", SCAN_DATE)
print("SCAN_TIMES", SCAN_TIMES)
print("BAR_TIMES", [bar_time(item).strftime("%H:%M") for item in SCAN_TIMES])

class ShoonyaWebApi:
    def __init__(self, user_id, token):
        self.user_id = str(user_id)
        self.token = token

    def _post(self, endpoint, jdata):
        values = dict(jdata)
        values.setdefault("uid", self.user_id)
        payload = ("jData=" + json.dumps(values, separators=(",", ":")) + "&jKey=" + self.token).encode("utf-8")
        request = urllib.request.Request(
            "https://trade.shoonya.com/NorenWClientWeb/" + endpoint,
            data=payload,
            headers={
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
                "Content-Type": "application/x-www-form-urlencoded",
            },
            method="POST",
        )
        try:
            with urllib.request.urlopen(request, timeout=20) as response:
                raw = response.read().decode("utf-8", errors="replace")
        except urllib.error.HTTPError as exc:
            raw = exc.read().decode("utf-8", errors="replace")
        return json.loads(raw) if raw.strip() else None

    def time_price_series(self, exchange, token, starttime, endtime, interval):
        jdata = {
            "ordersource": "API",
            "exch": exchange,
            "token": str(token),
            "st": str(int(starttime)),
            "intrv": str(interval),
        }
        if endtime is not None:
            jdata["et"] = str(int(endtime))
        response = self._post("TPSeries", jdata)
        return response if isinstance(response, list) else []

api = ShoonyaWebApi(config.USER_ID, SUPER_TOKEN)
universe = bot.load_universe()
print("UNIVERSE", len(universe))
print("SHOONYA_SESSION", config.USER_ID)

def fetch_shoonya_frame(record):
    columns = ["Timestamp", "Open", "High", "Low", "Close", "Volume"]
    response = api.time_price_series(
        exchange="NSE",
        token=record["token"],
        starttime=start_ep,
        endtime=end_ep,
        interval=15,
    )
    if not response:
        return pd.DataFrame(columns=columns)
    frame = pd.DataFrame(response)
    if "time" not in frame.columns:
        return pd.DataFrame(columns=columns)
    frame["Timestamp"] = pd.to_datetime(frame["time"], format="%d-%m-%Y %H:%M:%S", errors="coerce")
    for source, target in (("into", "Open"), ("inth", "High"), ("intl", "Low"), ("intc", "Close"), ("intv", "Volume")):
        if source in frame.columns:
            frame[target] = pd.to_numeric(frame[source], errors="coerce")
        else:
            frame[target] = np.nan
    clock = frame["Timestamp"].dt.strftime("%H:%M")
    frame = frame.loc[(clock >= "09:15") & (clock <= "15:15")]
    frame = frame.dropna(subset=["Timestamp", "Close", "Volume"])
    frame = frame.drop_duplicates(subset=["Timestamp"]).sort_values("Timestamp").reset_index(drop=True)
    return frame[columns]

print("FETCH_FYERS_START")
fyers_cache = fyers_data.refresh_fyers_cache(universe, days_back=10, max_workers=config.FYERS_WORKERS)
print("FYERS_CACHE", len(fyers_cache))

shoonya_cache = {}
for index, record in enumerate(universe, 1):
    frame = fetch_shoonya_frame(record)
    if not frame.empty:
        shoonya_cache[record["symbol"]] = frame
    if index % 50 == 0:
        print("SHOONYA_FETCHED", index, "of", len(universe), flush=True)
    time.sleep(0.05)

print("SHOONYA_CACHE", len(shoonya_cache))

fyers_features = {symbol: indicators.compute_features(frame) for symbol, frame in fyers_cache.items()}
shoonya_features = {symbol: indicators.compute_features(frame) for symbol, frame in shoonya_cache.items()}

def status_at(feature_map, symbol, timestamp):
    frame = feature_map.get(symbol)
    if frame is None or frame.empty:
        return "NO_DATA"
    completed = frame.loc[frame["Timestamp"] <= timestamp]
    if completed.empty or completed["Timestamp"].iloc[-1] != timestamp:
        return "STALE"
    return indicators.entry_status(completed)

def actionable(matches):
    return list(matches) if len(matches) <= config.MAX_SIGNALS_PER_TS else []

def joined(values):
    return ",".join(sorted(values)) if values else "NONE"

fyers_hits = {}
shoonya_hits = {}
for scan_hm in SCAN_TIMES:
    timestamp = bar_time(scan_hm)
    matched, _, _, _, _ = bot.evaluate_candidates(fyers_cache, universe, timestamp)
    fyers_hits[scan_hm] = sorted(matched)
    current_hits = []
    for record in universe:
        symbol = record["symbol"]
        if status_at(shoonya_features, symbol, timestamp) == "PASS":
            current_hits.append(symbol)
    shoonya_hits[scan_hm] = sorted(current_hits)

deviation_rows = []
for scan_hm in SCAN_TIMES:
    shoonya_set = set(shoonya_hits[scan_hm])
    fyers_set = set(fyers_hits[scan_hm])
    shoonya_action = set(actionable(shoonya_hits[scan_hm]))
    fyers_action = set(actionable(fyers_hits[scan_hm]))
    deviation_rows.append({
        "Scan": scan_hm,
        "Bar": bar_time(scan_hm).strftime("%Y-%m-%d %H:%M"),
        "Shoonya": joined(shoonya_set),
        "FYERS": joined(fyers_set),
        "Only_Shoonya": joined(shoonya_set - fyers_set),
        "Only_FYERS": joined(fyers_set - shoonya_set),
        "Shoonya_Actionable": joined(shoonya_action),
        "FYERS_Actionable": joined(fyers_action),
        "Exact_Raw_Match": shoonya_set == fyers_set,
        "Exact_Actionable_Match": shoonya_action == fyers_action,
        "Shoonya_Count": len(shoonya_set),
        "FYERS_Count": len(fyers_set),
    })

deviation_df = pd.DataFrame(deviation_rows)
print("DEVIATION_SUMMARY")
display(deviation_df)

compare_cols = ["Open", "High", "Low", "Close", "Volume", "UBB", "RSI", "ADX", "VWAP", "VSMA"]

def snapshot(feature_map, symbol, timestamp):
    frame = feature_map.get(symbol)
    if frame is None or frame.empty:
        return {}
    row = frame.loc[frame["Timestamp"] == timestamp]
    if row.empty:
        return {}
    return {column: row[column].iloc[-1] for column in compare_cols}

def numeric_diff(left, right):
    try:
        return float(right) - float(left)
    except (TypeError, ValueError):
        return np.nan

detail_rows = []
for scan_hm in SCAN_TIMES:
    timestamp = bar_time(scan_hm)
    previous = timestamp - timedelta(minutes=15)
    for symbol in sorted(set(shoonya_hits[scan_hm]) ^ set(fyers_hits[scan_hm])):
        fyers_now = snapshot(fyers_features, symbol, timestamp)
        shoonya_now = snapshot(shoonya_features, symbol, timestamp)
        fyers_prev = snapshot(fyers_features, symbol, previous)
        shoonya_prev = snapshot(shoonya_features, symbol, previous)
        row = {
            "Scan": scan_hm,
            "Bar": timestamp.strftime("%Y-%m-%d %H:%M"),
            "Symbol": symbol,
            "FYERS_Status": status_at(fyers_features, symbol, timestamp),
            "Shoonya_Status": status_at(shoonya_features, symbol, timestamp),
        }
        for column in compare_cols:
            fyers_value = fyers_now.get(column, np.nan)
            shoonya_value = shoonya_now.get(column, np.nan)
            row["FYERS_%s" % column] = fyers_value
            row["Shoonya_%s" % column] = shoonya_value
            row["%s_Diff" % column] = numeric_diff(fyers_value, shoonya_value)
            fyers_prior = fyers_prev.get(column, np.nan)
            shoonya_prior = shoonya_prev.get(column, np.nan)
            row["FYERS_Prev_%s" % column] = fyers_prior
            row["Shoonya_Prev_%s" % column] = shoonya_prior
            row["Prev_%s_Diff" % column] = numeric_diff(fyers_prior, shoonya_prior)
        detail_rows.append(row)

detail_df = pd.DataFrame(detail_rows)
if detail_df.empty:
    print("NO_BAR_LEVEL_DEVIATIONS")
else:
    compact_cols = [
        "Scan", "Bar", "Symbol", "FYERS_Status", "Shoonya_Status",
        "Close_Diff", "Volume_Diff", "UBB_Diff", "RSI_Diff", "ADX_Diff", "VWAP_Diff",
        "Prev_Close_Diff", "Prev_Volume_Diff", "Prev_UBB_Diff", "Prev_RSI_Diff", "Prev_ADX_Diff", "Prev_VWAP_Diff",
    ]
    print("BAR_LEVEL_DEVIATIONS")
    display(detail_df[compact_cols])
    display(detail_df)

summary_rows = []
for row in deviation_df.itertuples(index=False):
    summary_rows.append({
        "Scan": row.Scan,
        "Raw_Mismatch": not row.Exact_Raw_Match,
        "Actionable_Mismatch": not row.Exact_Actionable_Match,
        "Only_Shoonya_Count": 0 if row.Only_Shoonya == "NONE" else len(row.Only_Shoonya.split(",")),
        "Only_FYERS_Count": 0 if row.Only_FYERS == "NONE" else len(row.Only_FYERS.split(",")),
    })

summary_df = pd.DataFrame(summary_rows)
print("MISMATCH_COUNT_SUMMARY")
display(summary_df)
print("RAW_MISMATCH_SLOTS", int(summary_df["Raw_Mismatch"].sum()))
print("ACTIONABLE_MISMATCH_SLOTS", int(summary_df["Actionable_Mismatch"].sum()))
print("SINGLE_CELL_CHECK_DONE")